<a href="https://colab.research.google.com/github/Decoding-Data-Science/sharjah/blob/main/notebooks/2nd_D1_02_Documents_Understanding_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 1, Lab 2: Understand the policy documents

**Data and AI with Gemini | Corporate Workshop | all documents are fictional**

The database is **structured** data. The 12 policy documents are **unstructured** text, and in real life they are usually **PDF files**. In this notebook you will use Python to:

1. Read the documents (PDF or text) and turn them into a small table
2. See their structure: sections and clauses
3. Compare three versions of the same rule (approved, draft, superseded)
4. Search them, count words, and see how documents refer to each other
5. Build a **document dashboard**

Run each cell with **Shift + Enter**.

## Step 1. Load the documents

In [ ]:
# pypdf is a small library that reads the text out of PDF files.
!pip install -q pypdf

import io                           # to treat uploaded bytes like a file
import re                           # for finding patterns in text
import pandas as pd
import matplotlib.pyplot as plt
import importlib.metadata
from pypdf import PdfReader

print("pandas version:", pd.__version__)
print("pypdf version: ", importlib.metadata.version("pypdf"))

In [ ]:
# Upload the 12 documents. In the participant pack, open the folder 02_Documents.
# Choose ALL 12 PDF files (folder "pdf")  OR  all 12 text files (.txt). Not both.
from google.colab import files

uploaded = files.upload()

docs = {}          # file name  ->  the text of the document
first_lines = {}   # file name  ->  the top line (it holds the number, version and status)

for name, data in uploaded.items():
    if name.lower().endswith(".pdf"):
        # A PDF is not plain text: we ask pypdf to extract the text of every page.
        reader = PdfReader(io.BytesIO(data))
        text = "\n".join(page.extract_text() for page in reader.pages)
    elif name.lower().endswith(".txt"):
        text = data.decode("utf-8")
    else:
        continue
    first_lines[name] = text.strip().splitlines()[0]
    # A PDF repeats a header and a page number on every page. Remove them.
    lines = [l for l in text.splitlines()
             if not l.startswith("General International Airport (fictional) |") and not re.fullmatch(r"Page \d+", l.strip())]
    docs[name] = "\n".join(lines)

print(len(docs), "documents loaded")
if len(docs) != 12:
    print("Expected 12. If you uploaded both PDF and TXT files, run this cell again with only one kind.")
for name in sorted(docs):
    print("  ", name)

## Step 2. A first look

Open one document and read its first lines. Notice the **header** (number, version, status, owner) and the numbered **sections and clauses**.

In [ ]:
# A small helper: find a document by its number, for example "GIA-POL-01".
def get_text(doc_id):
    for name, text in docs.items():
        if name.startswith(doc_id):
            return text
    return ""

print(get_text("GIA-POL-01")[:1800])

## Step 3. Turn text into a table

We will read the header of every document and count its parts. This is how **unstructured text becomes structured data**.

A **clause** is one numbered rule, like `2.3`. Because a PDF wraps lines, we first glue each clause back into one piece.

In [ ]:
def clauses_of(text):
    """Cut a document into its numbered clauses (2.3, 3.1 ...). Each clause becomes one string."""
    found = re.findall(r"(?ms)^(\d+\.\d+ .*?)(?=^\d+\.\d+ |^Section \d+\.|\Z)", text)
    return [" ".join(c.split()) for c in found]

rows = []
for name, text in docs.items():
    doc_id = re.match(r"(GIA-[A-Z]+-\d+)", name).group(1)

    # The status is written on the top line of the document.
    top = first_lines[name].upper()
    if "SUPERSEDED" in top:
        status = "SUPERSEDED"
    elif "DRAFT" in top:
        status = "DRAFT"
    else:
        status = "APPROVED"

    words = len(text.split())                                              # how many words
    sections = len(re.findall(r"(?m)^Section \d+\.", text))                # lines starting "Section 1."
    clauses = len(clauses_of(text))                                        # numbered clauses
    rows.append([doc_id, status, words, sections, clauses])

summary = pd.DataFrame(rows, columns=["document", "status", "words", "sections", "clauses"]).sort_values("document").reset_index(drop=True)
summary

In [ ]:
# Chart: how long is each document? Colour = status.
colour = {"APPROVED": "#2e9e6b", "DRAFT": "#f39c12", "SUPERSEDED": "#d9534f"}
plt.figure(figsize=(11, 4.5))
plt.bar(summary["document"], summary["words"], color=[colour[s] for s in summary["status"]])
plt.xticks(rotation=60, ha="right")
plt.title("Words per document (green approved, orange draft, red superseded)")
plt.tight_layout()
plt.show()

**Your turn (1 minute):** Which document is the longest? How many are in force (approved)? Which two are *not* in force?

## Step 4. Three versions of the same rule

The airport has a limit for the average **departure immigration wait**. Three documents mention it. Only one is in force.

In [ ]:
for doc_id in ["GIA-POL-01", "GIA-DRAFT-11", "GIA-SUP-12"]:
    status = summary.loc[summary["document"] == doc_id, "status"].values[0]
    print(f"--- {doc_id}  ({status})")
    for clause in clauses_of(get_text(doc_id)):
        text = clause.lower()
        if "immigration" in text and "minutes" in text and "departure" in text:
            print("   ", clause[:230])
    print()

**Which limit is in force?** Only the **approved** document counts. A draft is a proposal. A superseded document is history. Remember this when a model quotes a number: it must quote the *approved* one.

## Step 5. Search the documents

In [ ]:
# A tiny search engine: find every clause that contains a word.
def search(word, only_status=None):
    found = []
    for name, text in docs.items():
        doc_id = re.match(r"(GIA-[A-Z]+-\d+)", name).group(1)
        status = summary.loc[summary["document"] == doc_id, "status"].values[0]
        if only_status and status != only_status:
            continue
        for clause in clauses_of(text):
            if word.lower() in clause.lower():
                found.append([doc_id, status, clause[:150]])
    return pd.DataFrame(found, columns=["document", "status", "clause"])

search("exception")

In [ ]:
# The same search, but only inside approved documents.
search("exception", only_status="APPROVED")

## Step 6. Which words matter?

In [ ]:
# Count the most common words in the approved documents. We skip very common words.
skip = set("the a an of to in and or for on at be is are was by with that this it as if any must not no may from shall their which than".split())
approved_ids = set(summary.loc[summary["status"] == "APPROVED", "document"])
approved_text = " ".join(text for name, text in docs.items() if re.match(r"(GIA-[A-Z]+-\d+)", name).group(1) in approved_ids)
words = [w for w in re.findall(r"[a-z]+", approved_text.lower()) if w not in skip and len(w) > 3]
top = pd.Series(words).value_counts().head(15)
top

## Step 7. How do documents refer to each other?

In [ ]:
# Count how many times each document mentions another document number (for example "GIA-STD-03").
ids = list(summary["document"])
refs = pd.DataFrame(0, index=ids, columns=ids)
for name, text in docs.items():
    this_id = re.match(r"(GIA-[A-Z]+-\d+)", name).group(1)
    for other in ids:
        if other != this_id:
            refs.loc[this_id, other] = text.count(other)

plt.figure(figsize=(8, 6.5))
plt.imshow(refs.values, cmap="Blues")
plt.xticks(range(len(ids)), ids, rotation=70, ha="right")
plt.yticks(range(len(ids)), ids)
plt.colorbar(label="mentions")
plt.title("Who mentions whom (row mentions column)")
plt.tight_layout()
plt.show()

## Step 8. Your document dashboard

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Policy documents dashboard (fictional)", fontsize=16, fontweight="bold")

# 1. words per document
ax[0, 0].bar(summary["document"], summary["words"], color=[colour[s] for s in summary["status"]])
ax[0, 0].set_title("Words per document")
ax[0, 0].tick_params(axis="x", rotation=70)

# 2. how many documents per status
counts = summary["status"].value_counts()
ax[0, 1].bar(counts.index, counts.values, color=[colour[s] for s in counts.index])
ax[0, 1].set_title("Documents by status")

# 3. clauses per document
ax[1, 0].bar(summary["document"], summary["clauses"], color="#2b8cbe")
ax[1, 0].set_title("Numbered clauses per document")
ax[1, 0].tick_params(axis="x", rotation=70)

# 4. top words
ax[1, 1].barh(top.index[::-1], top.values[::-1], color="#7fb3d5")
ax[1, 1].set_title("Most common words (approved documents)")

plt.tight_layout()
plt.show()

## Your turn

Pick **one**:

1. Search for **"45 minutes"** and **"30 minutes"**. Which documents use these numbers?
2. Use `search("counter")` and list the minimum number of counters each document asks for.
3. Which document is mentioned most by other documents? (Look at the heat map.)
4. Add a chart of **sections per document**.

### What you should be able to say after this notebook
* Documents are unstructured text, but they have structure we can find: headers, sections, clauses.
* A PDF needs one extra step (extract the text) before Python can analyse it.
* Python can turn a pile of documents into a table, search them, and count them.
* A document has a **status**. Only approved documents are rules in force.
* This is exactly the problem that RAG solves later: finding the right clause in the right document.

**Next notebook:** talk to Gemini. Change the model, set a system prompt, and add context in code.